## Accessing particle-type EAGLE data

This notebook is based on the examples in https://arxiv.org/pdf/1706.09899 

### Identifying the data we need
For this project, we will be using the EAGLE (Evolution and Assembly of GaLaxies and their Environments) cosmological simulations: https://astro.dur.ac.uk/ICC/Eagle/index.php. As we are interested in the cosmic star formation history SFRD(Z, z), we want to obtain the metallicity (Z) and star formation rate (SFR) of every star-forming gas cell in the simulation, at a given snapshot (at a specific redshift or lookback time). This is because we need to know the metallicity and rate that stars are forming at, at a given moment, to know what metallicity and rate binary black hole progenitors might be forming at. The *formation* metallicity is what will affect the evolution of the stars. This means we will need to obtain gas data, not star data.

EAGLE data comes in two different forms: the galaxy database (http://virgodb.dur.ac.uk:8080/Eagle/) and the particle data (https://dataweb.cosma.dur.ac.uk:8443/eagle-snapshots/). As we are interested in the *cosmic* star formation history, we do not need to keep track of which galaxies what star formation is happening in.
This means that we will need to access the *particle data*.

The EAGLE documentation is in this paper: https://arxiv.org/pdf/1706.09899. Go to Table 3 and identify which Particle Type corresponds to gas cells.

Let's first read in some packages we will use. Note that for the cosmology, we will use the Planck Collaboration et al. 2014 (https://arxiv.org/pdf/1303.5062) cosmological parameters, which you can easily read in with astrop (listed as Planck13 in astropy). This is because EAGLE uses the Planck 2014 cosmology, so we will want to be consistent with that throughout.

If you do not have any of these packages, pip install them in the command line.

In [4]:
import h5py as h5
import numpy as np
import astropy.units as u
from astropy.cosmology import Planck13 as cosmo #set cosmology to match what is used by EAGLE
import matplotlib.pyplot as plt

### Downloading data
To download EAGLE data, go to the EAGLE Public Data Release page: https://astro.dur.ac.uk/ICC/Eagle/database.php. 
For this tutorial, you will be using the small test volume: RefL0012N0188 and we will be downloading a single snapshot (snapshot 28, at z=0).

On the data download page, you will see an option to download the data directly. We will instead use the wget command, an example of which is shown on that page. This is because when you will be downloading the actual data for your project, you will be downloading large volumes of it using a python script.

In [ ]:
#This is the command you will use. The username and password are in the project Google Doc. 
# The "!" before the command is what lets you run a command line command directly in this jupyter notebook 
# we will do this differently when downloading data using a python script, but for simplicity let's do this now

! wget --user=[username] --password=[password] --content-disposition "https://dataweb.cosma.dur.ac.uk:8443/eagle-snapshots//download?run=RefL0012N0188&snapnum=28"

In [ ]:
# Unzip the file and get the path to the snapshot_028_z000p000 directory (the innermost one)

path = ""

In [1]:
#This is an example read_dataset function from the EAGLE documentation paper (Section 4.1), to read in some data 

def read_dataset(path, itype, att, nfiles = 16):
    """ Read a selected dataset, itype is the PartType and att is the attribute name """

    # Output array
    #data = []
    # Loop over each file and extract the data
    f = h5.File(path+'/snap_028_z000p000.%i.hdf5'%i, 'r')
    data = f['PartType%i/%s'%(itype, att)][...]
    #data.append(tmp)

    # Get conversion factors
    aexp = f['PartType%i/%s'%(itype, att)].attrs.get('aexp-scale-exponent')
    hexp = f['PartType%i/%s'%(itype, att)].attrs.get('h-scale-exponent')
        
    # Get expansion factor and Hubble parameter from the header
    a = f['Header'].attrs.get('Time')
    h = f['Header'].attrs.get('HubbleParam')

    f.close()

    # Convert to physical
    if data.dtype != np.int32 and data.dtype != np.int64:
        data = np.multiply(data, a**aexp * h**hexp, dtype='f8')
                            
    return data

In [2]:
# Try using read_dataset to read in the metallicity from one of the files
itype = 0 #PartType0 = gas
att = 'Metallicity'
Z = read_dataset(path, itype, att, nfiles = 1) 

NameError: name 'path' is not defined

In [3]:
#Lets look at the metallicity distribution by plotting it as a histogram. You can also print the bins to look at them

nBins = 60 #let's use 60 metallicity bins
Zbins = np.logspace(-10, 0, nBins+1) #logarithmically spaced bins between Z=1e-10 and Z=1e0=1
plt.hist(Z, bins= Zbins);
plt.xscale('log')

NameError: name 'np' is not defined

In [ ]:
# Now try repeating the same but for the star formation rate (you can choose the bins, and change x/y axis scaling as necessary to see the data). 
# You can refer to Table 4 in the EAGLE documentation paper



In [ ]:
# Now let's try getting the quantity we are actually interested in: the SFR binned into metallicity bins
# Getting this quantity for all snapshots and putting it into a big 2D array (where each row will be the redshift/snapshot)

data, e = np.histogram(Z, bins=Zbins, weights=SFR) #use the SFR you got in the previous step

#Plot the histogram data (we are using np histogram so that we can do the weights. np.histogram doesn't automatically give you the plot)
plt.hist(data, bins= Zbins);
plt.xscale('log') #see if the y axis should also be log scale!

In [ ]:
#This is an example read_header function from the EAGLE documentation paper (Section 4.2). 
#The header has broad information for the simulation and snapshot, such as the redshift of the snapshot, and box size of the simulation
#I think the "time" is relative time to the total time

def read_header(path):
    """Read various attributes from the header group """
    f = h5.File(path+'/snap_028_z000p000.0.hdf5', 'r')
    z = f['Header'].attrs.get('Redshift') # redshift
    a = f['Header'].attrs.get('Time') # scale factor
    h = f['Header'].attrs.get('HubbleParam') # h
    boxsize = f['Header']. attrs.get('BoxSize') # L [cMpc/h]
    f.close()

    return z, a, h, boxsize

In [ ]:
z, a, h, boxsize = read_header(path)
lookbacktime = cosmo.lookback_time(z).value #calculate lookback time using appropriate cosmology

print("The redshift of this snapshot is z=%.2f"%z)
print("The lookback time of this snapshot is %.2f"%lookbacktime, 'Gyr') #0 = present day!
print("The box size is", boxsize*h, "cMpc") #for this simulation, it should be 12.5 cMpc... not sure why this isn't (unit conversions?)